# 04 - Sklearn base models (TF-IDF + LogReg + CalibratedClassifierCV(LinearSVC))

5-fold OOF + full-train refit. Outputs:

* `artifacts/probs/logreg_{oof,val,test}.npy` and `artifacts/metrics/logreg.json`
* `artifacts/probs/svc_{oof,val,test}.npy` and `artifacts/metrics/svc.json`

In [ ]:
%pip install -q scikit-learn pandas numpy

In [ ]:
import sys, os, time, json

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

from tm_research.ensemble.utils_io import load_splits, save_probs, save_metrics
from tm_research.ensemble.utils_stacking import kfold_oof_probs

train_df, val_df, test_df, label_map = load_splits()
y_train = train_df['label'].map(label_map.label2id).to_numpy()
y_val = val_df['label'].map(label_map.label2id).to_numpy()
y_test = test_df['label'].map(label_map.label2id).to_numpy()
X_train = train_df['text'].tolist()
X_val = val_df['text'].tolist()
X_test = test_df['text'].tolist()
print(len(X_train), len(X_val), len(X_test), label_map.num_classes)

## Pipeline factories

Both classifiers use the same TF-IDF (1-2grams, min_df=2). LinearSVC is wrapped in `CalibratedClassifierCV(method='sigmoid', cv=3)` so we get probabilities.

In [ ]:
def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def make_logreg(seed=42):
    return Pipeline([
        ('tfidf', tfidf()),
        ('clf', LogisticRegression(
            max_iter=2000, C=1.0, n_jobs=-1, solver='lbfgs', random_state=seed
        )),
    ])

def make_svc(seed=42):
    base_svc = LinearSVC(C=1.0, random_state=seed)
    return Pipeline([
        ('tfidf', tfidf()),
        ('clf', CalibratedClassifierCV(base_svc, method='sigmoid', cv=3)),
    ])

MODELS = {
    'logreg': make_logreg,
    'svc': make_svc,
}
SEED = 42
N_FOLDS = 5

## Run OOF + full-train refit for each model

In [ ]:
for name, factory in MODELS.items():
    print(f'=== {name} ===')
    t0 = time.time()

    def fit_predict(x_tr, y_tr, x_va, _factory=factory):
        pipe = _factory(seed=SEED)
        pipe.fit(list(x_tr), y_tr)
        return pipe.predict_proba(list(x_va))

    oof, fold_accs = kfold_oof_probs(
        fit_predict, X_train, y_train,
        num_classes=label_map.num_classes, n_splits=N_FOLDS, seed=SEED,
    )
    save_probs(name, 'oof', oof)
    oof_acc = float((oof.argmax(1) == y_train).mean())

    pipe = factory(seed=SEED)
    pipe.fit(X_train, y_train)
    val_probs = pipe.predict_proba(X_val)
    test_probs = pipe.predict_proba(X_test)
    save_probs(name, 'val', val_probs)
    save_probs(name, 'test', test_probs)

    val_acc = float((val_probs.argmax(1) == y_val).mean())
    test_acc = float((test_probs.argmax(1) == y_test).mean())
    metrics = {
        'model_name': name,
        'output_name': name,
        'seed': SEED,
        'n_folds': N_FOLDS,
        'fold_val_accuracies': fold_accs,
        'oof_accuracy': oof_acc,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'num_train': int(len(X_train)),
        'num_val': int(len(X_val)),
        'num_test': int(len(X_test)),
        'elapsed_seconds': time.time() - t0,
    }
    save_metrics(name, metrics)
    print(json.dumps(metrics, indent=2))

In [ ]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()